# Canadian Crime Statistics Data Cleaning

This notebook loads **Statistics Canada Crime Severity Index (CSI)** data for 2020–2024, cleans and standardizes it, and prepares it for **exploratory data analysis (EDA)** and merging with **housing affordability (CMHC) data**.

## Step 1: Import Libraries
I will use pandas for dataframes and numpy for numerical operations.

In [6]:
import pandas as pd # for loading and manipulating .csv files
import numpy as np  # for calculations and handling NaN values

## Step 2: File Path + City Name List
CMHC calls cities things like `"Halifax CMA"`, but StatCan calls the same city `"Halifax, Nova Scotia"` (plus a bracketed city code I have to strip off). So I built a dictionary mapping one to the other. Had to actually open and check the raw file spelling for a few of these. Ottawa-Gatineau turned out to be split into an Ontario and a Quebec part in the crime data, which conveniently matches how CMHC splits it.

In [7]:
years = range(2020, 2025)
csi_file_path = "../data/raw/csi_canada_cma_2020-2024.csv"

# mapping CMHC centre names -> StatCan GEO names
centre_to_geo = {
    "Calgary CMA": "Calgary, Alberta",
    "Edmonton CMA": "Edmonton, Alberta",
    "Halifax CMA": "Halifax, Nova Scotia",
    "Hamilton CMA": "Hamilton, Ontario",
    "Kelowna CMA": "Kelowna, British Columbia",
    "Kitchener-Cambridge-Waterloo CMA": "Kitchener-Cambridge-Waterloo, Ontario",
    "London CMA": "London, Ontario",
    "Montréal CMA": "Montréal, Quebec",
    "Ottawa-Gatineau CMA (Ont. part)": "Ottawa-Gatineau, Ontario part",
    "Ottawa-Gatineau CMA (Qué. part)": "Ottawa-Gatineau, Quebec part",
    "Québec CMA": "Québec, Quebec",
    "Regina CMA": "Regina, Saskatchewan",
    "Saskatoon CMA": "Saskatoon, Saskatchewan",
    "Toronto CMA": "Toronto, Ontario",
    "Vancouver CMA": "Vancouver, British Columbia",
    "Victoria CMA": "Victoria, British Columbia",
    "Winnipeg CMA": "Winnipeg, Manitoba",
}

# need this reversed for later
geo_to_centre = {geo: centre for centre, geo in centre_to_geo.items()} 

## Step 3: Read the Raw File and Clean Up the Geography Names
- Load data, strip city codes, and check that all 17 centres exist and match in the Crime Severity Index

In [8]:
csi_raw = pd.read_csv(csi_file_path, encoding="utf-8-sig")
print(csi_raw.shape)
print(csi_raw.head())

(23916, 15)
   REF_DATE     GEO           DGUID                          Statistics  \
0      1998  Canada  2021A000011124                Crime severity index   
1      1998  Canada  2021A000011124        Violent crime severity index   
2      1998  Canada  2021A000011124    Non-violent crime severity index   
3      1998  Canada  2021A000011124          Youth crime severity index   
4      1998  Canada  2021A000011124  Youth violent crime severity index   

     UOM  UOM_ID SCALAR_FACTOR  SCALAR_ID     VECTOR  COORDINATE   VALUE  \
0  Index     160         units          0  v44312461         1.1  118.84   
1  Index     160         units          0  v44312463         1.3   97.80   
2  Index     160         units          0  v44312465         1.5  126.93   
3  Index     160         units          0  v53079455         1.7  110.24   
4  Index     160         units          0  v53079457         1.9   86.59   

  STATUS  SYMBOL  TERMINATED  DECIMALS  
0    NaN     NaN         NaN         2 

In [9]:
# strip the trailing [city code] off the GEO column
csi_raw["geo_clean"] = csi_raw["GEO"].str.replace(r"\s*\[[^\]]*\]\s*$", "", regex=True)

# make sure I didn't typo any of the city names above
available_geo = set(csi_raw["geo_clean"].unique())
missing = {centre: geo for centre, geo in centre_to_geo.items() if geo not in available_geo}

if missing:
    print("Did not match, fix centre_to_geo:")
    for centre, geo in missing.items():
        print(f"  {centre} -> tried {geo}")
else:
    print("All 17 matched!")

All 17 matched!


## Step 4: Filter Down to the Cities and Years I Actually Need
The raw file has every province, territory, and CMA going back to 1998, so I am filtering down to just my 17 cities for 2020-2024.

In [10]:
csi_filtered = csi_raw[
    csi_raw["geo_clean"].isin(centre_to_geo.values()) & csi_raw["REF_DATE"].isin(years)
].copy()

csi_filtered["centre"] = csi_filtered["geo_clean"].map(geo_to_centre)
csi_filtered = csi_filtered.rename(columns={"REF_DATE": "year"})
csi_filtered["value"] = pd.to_numeric(csi_filtered["VALUE"], errors="coerce")

print(csi_filtered.shape)
csi_filtered[["centre", "year", "Statistics", "value"]].head()

(1530, 18)


,centre,year,Statistics,value
17958,Halifax CMA,2020,Crime severity index,62.30
17959,Halifax CMA,2020,Percent change in crime severity index,2.05
17960,Halifax CMA,2020,Violent crime severity index,85.07
17961,Halifax CMA,2020,Percent change in violent crime severity index,0.84
17962,Halifax CMA,2020,Non-violent crime severity index,53.96


## Step 5: Pivot It Wide
Right now, every city/year has 18 rows of statistics. I want one row per city/year with a column for each statistic, same shape as the CMHC data so they'll merge nicely later. Also, i'll abbreviate the statistic names to something shorter than "Percent change in non-violent weighted clearance rate."

In [12]:
csi_wide = csi_filtered.pivot_table(
    index=["centre", "year"],
    columns="Statistics",
    values="value",
    aggfunc="first",
).reset_index()

# flatten and convert the column names to snake_case
csi_wide.columns = [
    col if col in ["centre", "year"] else col.strip().lower().replace(" ", "_").replace("-", "_")
    for col in csi_wide.columns
]

# shortening names
rename_map = {
    "crime_severity_index": "csi_total",
    "percent_change_in_crime_severity_index": "csi_total_pct_change",
    "violent_crime_severity_index": "csi_violent",
    "percent_change_in_violent_crime_severity_index": "csi_violent_pct_change",
    "non_violent_crime_severity_index": "csi_nonviolent",
    "percent_change_in_non_violent_crime_severity_index": "csi_nonviolent_pct_change",
    "youth_crime_severity_index": "csi_youth",
    "percent_change_in_youth_crime_severity_index": "csi_youth_pct_change",
    "youth_violent_crime_severity_index": "csi_youth_violent",
    "percent_change_in_youth_violent_crime_severity_index": "csi_youth_violent_pct_change",
    "youth_non_violent_crime_severity_index": "csi_youth_nonviolent",
    "percent_change_in_youth_non_violent_crime_severity_index": "csi_youth_nonviolent_pct_change",
    "weighted_clearance_rate": "clearance_rate",
    "percent_change_in_weighted_clearance_rate": "clearance_rate_pct_change",
    "violent_weighted_clearance_rate": "clearance_rate_violent",
    "percent_change_in_violent_weighted_clearance_rate": "clearance_rate_violent_pct_change",
    "non_violent_weighted_clearance_rate": "clearance_rate_nonviolent",
    "percent_change_in_non_violent_weighted_clearance_rate": "clearance_rate_nonviolent_pct_change",
}
csi_clean = csi_wide.rename(columns=rename_map)
csi_clean = csi_clean.sort_values(["centre", "year"]).reset_index(drop=True)

print(csi_clean.shape)
csi_clean.head()

(85, 14)


,centre,year,csi_total,csi_nonviolent,clearance_rate_nonviolent,csi_total_pct_change,csi_nonviolent_pct_change,clearance_rate_nonviolent_pct_change,csi_violent_pct_change,clearance_rate_violent_pct_change,clearance_rate_pct_change,csi_violent,clearance_rate_violent,clearance_rate
0,Calgary CMA,2020,79.96,79.88,22.06,-15.02,-17.52,0.41,-7.16,8.35,5.51,79.64,47.24,28.70
1,Calgary CMA,2021,72.78,68.76,23.42,-8.98,-13.92,6.17,5.96,-5.91,5.71,84.39,44.45,30.34
2,Calgary CMA,2022,77.48,76.60,23.70,6.46,11.40,1.20,-3.63,6.59,1.38,81.33,47.38,30.76
3,Calgary CMA,2023,72.66,68.38,27.02,-6.22,-10.73,14.01,4.39,-3.36,8.06,84.90,45.79,33.24
4,Calgary CMA,2024,63.29,57.64,27.22,-12.90,-15.71,0.74,-7.21,7.62,5.32,78.78,49.28,35.01


## Step 6: Missing Values
- Interpolate first, then ffill/bfill whatever's left over. There isn't much compared to the CMHC file but if an alternate source is uploaded it's good to check there are no gaps.

In [17]:
num_cols = [c for c in csi_clean.columns if c not in ["centre", "year"]]

# check before
print(f"Before: \n{csi_clean.isna().sum()}")

csi_clean[num_cols] = (
    csi_clean.groupby("centre", group_keys=False)[num_cols]
    .apply(lambda g: g.interpolate(method="linear", limit_direction="both"))
)
csi_clean[num_cols] = csi_clean.groupby("centre")[num_cols].ffill().bfill()

# check after, should all be 0 now
print(f"\nAfter:\n{csi_clean.isna().sum()}")

# make sure value there are no null values
assert csi_clean.isna().sum().sum() == 0
# make sure nobody got dropped along the way
assert csi_clean["centre"].nunique() == 17

Before: 
centre                                  0
year                                    0
csi_total                               0
csi_nonviolent                          0
clearance_rate_nonviolent               0
csi_total_pct_change                    0
csi_nonviolent_pct_change               0
clearance_rate_nonviolent_pct_change    0
csi_violent_pct_change                  0
clearance_rate_violent_pct_change       0
clearance_rate_pct_change               0
csi_violent                             0
clearance_rate_violent                  0
clearance_rate                          0
dtype: int64

After:
centre                                  0
year                                    0
csi_total                               0
csi_nonviolent                          0
clearance_rate_nonviolent               0
csi_total_pct_change                    0
csi_nonviolent_pct_change               0
clearance_rate_nonviolent_pct_change    0
csi_violent_pct_change                  0
clea

## Step 7: Save It

In [18]:
csi_clean.to_csv("../data/processed/csi_clean_2020_2024.csv", index=False)
print("Saved:", csi_clean.shape)

Saved: (85, 14)
